# Lab: from raw Titanic rows to a saved model

Companion notebook of the lesson *Pipelines* — https://insper.github.io/ML/2026.2/classes/pipelines/

Goal: turn the raw Titanic table into one pipeline (preprocessing + a black-box k-NN classifier), fit it on the training rows, check what every step learned, predict on new raw passengers, save and reload it, and extend it with a transformer of your own. Run the cells in order, in one session.

The **fundamental part** is steps 1 to 5; the **advanced part**, steps 6 to 9. Every step starts with a **Predict**: answer it *before* running the cell. The **Simulations** ask for a small experiment of your own in the empty cell below them; the solutions are on the lesson page.

## Fundamental part — steps 1 to 5

### Step 1 — load and look

**Predict.** Which columns of a raw passenger list do you expect to have holes? Which ones could a model not use as they are?

**Resposta (antes de rodar):** Espero buracos em **Age** (bastante, talvez ~20% das linhas), **Cabin**
(muitíssimos, a maioria dos passageiros de 3ª classe não tem cabine registrada) e **Embarked** (pouquíssimos,
talvez 1 ou 2 linhas). Colunas que um modelo não consegue usar como estão, sem nenhum pré-processamento:
`Name`, `Sex`, `Ticket`, `Cabin` e `Embarked` são texto (precisam virar número); `Age` e `Fare` são numéricas
mas têm buracos, então também não podem ir direto para um k-NN sem imputação.

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
print("missing:", df.isna().sum()[lambda s: s > 0].to_dict())
print(df[['Name', 'Sex', 'Age', 'Fare', 'Cabin', 'Embarked']].head(3).to_string())

(891, 12)
missing: {'Age': 177, 'Cabin': 687, 'Embarked': 2}
                                                  Name     Sex   Age     Fare Cabin Embarked
0                              Braund, Mr. Owen Harris    male  22.0   7.2500   NaN        S
1  Cumings, Mrs. John Bradley (Florence Briggs Thayer)  female  38.0  71.2833   C85        C
2                               Heikkinen, Miss. Laina  female  26.0   7.9250   NaN        S


**Resposta (depois de rodar):** Confirmado: `Age` tem 177 valores faltantes, `Cabin` tem 687 (a
grande maioria das 891 linhas) e `Embarked` tem só 2 — exatamente a ordem de grandeza que eu esperava. Olhando
as 3 primeiras linhas, `Name`, `Sex` e `Embarked` são texto puro, e `Cabin` já aparece como `NaN` na 1ª e na
3ª linha. Bate exatamente com o publicado na página (Age 177, Cabin 687, Embarked 2).

### Step 2 — split once

**Predict.** With `stratify=y`, how close will the survival rates of the training and test rows be? And why keep in `X` the columns the model will never use, such as `Name` and `Ticket`?

**Resposta (antes de rodar):** Com `stratify=y`, a taxa de sobrevivência do treino e a do teste devem
ficar muito próximas — a estratificação força a mesma proporção da classe em ambos os conjuntos, a menos de um
arredondamento de poucas centenas de uma unidade. Manter `Name` e `Ticket` em `X`, mesmo que nenhum branch do
pipeline os use: o split é feito sobre as linhas, não sobre as colunas que o modelo vai consumir — quem decide
o que entra no modelo é o `ColumnTransformer`, não o `train_test_split`. Guardar essas colunas dá a opção de
construir features novas a partir delas depois (como o título extraído de `Name` no Step 9) sem ter que refazer
o split e arriscar misturar treino e teste.

In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='Survived')              # all 11 raw columns, text and holes included
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"survival rate: train {y_train.mean():.3f} · test {y_test.mean():.3f}")

(712, 11) (179, 11)
survival rate: train 0.383 · test 0.385


**Resposta (depois de rodar):** Confirmado: a taxa de sobrevivência ficou em 0.383 no treino e
0.385 no teste — uma diferença de só 0.002, bem dentro do esperado pela estratificação. `X_train`/`X_test`
têm (712, 11) e (179, 11): 11 colunas, porque `Survived` já foi retirado de `X`. `Name` e `Ticket` continuam
lá, disponíveis para qualquer feature futura (como o título do Step 9), mesmo que nenhum branch do
`ColumnTransformer` os use hoje.

### Step 3 — the preprocessing, column by column

**Predict.** Which of the 11 raw columns will reach the model, and what happens to the ones no branch mentions? Has anything been fit when this cell ends?

**Resposta (antes de rodar):** Chegam ao modelo as colunas citadas nos três branches: `Age`, `SibSp`,
`Parch` (branch `num`), `Fare` (branch `fare`), `Pclass`, `Sex`, `Embarked` (branch `cat`) — 7 das 11 colunas
brutas. As que nenhum branch menciona (`PassengerId`, `Name`, `Ticket`, `Cabin`) são descartadas, porque o
comportamento padrão do `ColumnTransformer` é `remainder='drop'`. Nada foi ajustado (`fit`) ainda ao final
desta célula: construir um `Pipeline`/`ColumnTransformer` só declara a estrutura; é `model.fit(...)` que
dispara o aprendizado.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ('num',  make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
             ['Age', 'SibSp', 'Parch']),
    ('fare', make_pipeline(SimpleImputer(strategy='median'),
                           FunctionTransformer(np.log1p, feature_names_out='one-to-one'),
                           StandardScaler()),
             ['Fare']),
    ('cat',  make_pipeline(SimpleImputer(strategy='most_frequent'),
                           OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
             ['Pclass', 'Sex', 'Embarked']),
])
model = Pipeline([('prep', preprocess), ('knn', KNeighborsClassifier())])

used = [c for _, _, cols in preprocess.transformers for c in cols]
for name, _, cols in preprocess.transformers:
    print(f"{name:4s} <- {cols}")
print("dropped by remainder='drop':", [c for c in X.columns if c not in used])

num  <- ['Age', 'SibSp', 'Parch']
fare <- ['Fare']
cat  <- ['Pclass', 'Sex', 'Embarked']
dropped by remainder='drop': ['PassengerId', 'Name', 'Ticket', 'Cabin']


**Resposta (depois de rodar):** Exatamente como previsto: `num <- ['Age', 'SibSp', 'Parch']`,
`fare <- ['Fare']`, `cat <- ['Pclass', 'Sex', 'Embarked']` — 7 colunas usadas — e
`dropped by remainder='drop': ['PassengerId', 'Name', 'Ticket', 'Cabin']`, as 4 que nenhum branch cita. A
célula só imprime a estrutura dos branches; nenhum número foi aprendido ainda (confirma a hipótese — não há
nenhuma chamada a `.fit` aqui, só a montagem dos objetos).

### Step 4 — fit on train, score on test

**Predict.** How many separate `fit` calls does the single `model.fit` trigger? Will a k-NN with its default settings beat the "always did not survive" baseline by a lot or by a little?

**Resposta (antes de rodar):** Um único `model.fit` deve disparar várias chamadas de `fit` internas —
uma por transformador que aprende algo em cada branch (2 imputers + 1 scaler no `num`, 1 imputer + 1 scaler no
`fare` — o `log1p` não aprende nada — e 1 imputer + 1 one-hot no `cat`), mais o `fit` final do
`KNeighborsClassifier`, algo como 7-8 chamadas no total. Sobre o k-NN: como a taxa de sobrevivência é
desbalanceada (bem menos da metade sobrevive), a baseline "sempre não sobreviveu" já acerta a maioria das
vezes; espero que o k-NN vença por uma margem moderada (uns 15-20 pontos percentuais), não por um salto
enorme.

In [4]:
from sklearn.dummy import DummyClassifier

model.fit(X_train, y_train)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"test accuracy · pipeline {model.score(X_test, y_test):.3f} · "
      f"always 'did not survive' {baseline.score(X_test, y_test):.3f}")

test accuracy · pipeline 0.782 · always 'did not survive' 0.615


**Resposta (depois de rodar):** `model.fit` deu acurácia de teste **0.782** contra **0.615** da
baseline "sempre não sobreviveu" — uma melhora de 16.7 pontos percentuais, na faixa moderada que eu esperava
(nem gigantesca, nem desprezível). A contagem de chamadas de `fit` é consistente com a hipótese: o Step 5
confirma que o imputer e o scaler do branch `num`, o imputer e o scaler do `fare`, e o imputer + encoder do
`cat` guardam estatísticas aprendidas — 2+1 no `num`, 1+1 no `fare`, 1+1 no `cat`, mais 1 do `KNeighborsClassifier`,
soma 7 chamadas de `fit` internas dentro da única chamada a `model.fit`.

**Simulation: the same recipe without the scalers.** Build a second `ColumnTransformer` in which the `'num'` and `'fare'` branches keep only their imputer (no logarithm, no scaler), put it in a pipeline with the same `KNeighborsClassifier()`, fit it on the training rows and compare its test accuracy with the 0.782 of the pipeline. Which column dominates the distances when nothing is scaled?

In [5]:
# Simulation: a mesma receita sem os scalers
preprocess_noscale = ColumnTransformer([
    ('num',  SimpleImputer(strategy='median'), ['Age', 'SibSp', 'Parch']),
    ('fare', SimpleImputer(strategy='median'), ['Fare']),
    ('cat',  make_pipeline(SimpleImputer(strategy='most_frequent'),
                           OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
             ['Pclass', 'Sex', 'Embarked']),
])
model_noscale = Pipeline([('prep', preprocess_noscale), ('knn', KNeighborsClassifier())])
model_noscale.fit(X_train, y_train)

acc_scaled = model.score(X_test, y_test)
acc_noscale = model_noscale.score(X_test, y_test)
print(f"test accuracy · com scalers {acc_scaled:.3f} · sem scalers {acc_noscale:.3f}")
print("faixa de Fare no treino (bruto):   ", X_train['Fare'].min(), "-", X_train['Fare'].max())
print("faixa de Age no treino (bruto):    ", X_train['Age'].min(), "-", X_train['Age'].max())
print("faixa de SibSp no treino (bruto):  ", X_train['SibSp'].min(), "-", X_train['SibSp'].max())
print("faixa de Parch no treino (bruto):  ", X_train['Parch'].min(), "-", X_train['Parch'].max())

test accuracy · com scalers 0.782 · sem scalers 0.676
faixa de Fare no treino (bruto):    0.0 - 512.3292
faixa de Age no treino (bruto):     0.42 - 80.0
faixa de SibSp no treino (bruto):   0 - 8
faixa de Parch no treino (bruto):   0 - 6


**Resposta:** Sem os scalers a acurácia cai de 0.782 para **0.676** — mais de 10 pontos
percentuais, confirmando que o k-NN sofre bastante com features em escalas diferentes. A faixa bruta de
`Fare` no treino é 0.0–512.33, muito maior que a de `Age` (0.42–80.0), `SibSp` (0–8) ou `Parch` (0–6): sem
padronização, é a `Fare` que domina a distância euclidiana do k-NN, porque sua escala numérica é ordens de
grandeza maior que a das outras colunas.

### Step 5 — what each step learned

**Predict.** The median age of all 891 passengers is 28.0. Has the imputer inside the pipeline learned 28.0, or another number? And how many rows has the scaler seen?

**Resposta (antes de rodar):** O imputer de dentro do pipeline só viu `X_train` (712 das 891 linhas,
80% do total), então ele deve ter aprendido a mediana calculada **só com o treino** — um valor provavelmente
próximo de 28.0, mas não necessariamente igual, já que é uma mediana sobre uma amostra diferente (menor) da
coluna completa. O scaler deve ter visto exatamente as mesmas 712 linhas de treino que o imputer viu, nunca
nenhuma linha de teste.

In [6]:
prep = model.named_steps['prep']
num_imputer, num_scaler = prep.named_transformers_['num']
print("Age, SibSp, Parch medians:", num_imputer.statistics_)
print("means after imputing:     ", num_scaler.mean_.round(3))
print("standard deviations:      ", num_scaler.scale_.round(3))
print("rows the scaler saw:      ", num_scaler.n_samples_seen_)
print("Age median, train only vs all 891 rows:", X_train['Age'].median(), "vs", X['Age'].median())
fare_imputer, _, fare_scaler = prep.named_transformers_['fare']
print(f"Fare: median {fare_imputer.statistics_[0]:.2f} · mean of log1p(Fare) {fare_scaler.mean_[0]:.3f}")
print("categories:", [c.tolist() for c in prep.named_transformers_['cat'][-1].categories_])

Age, SibSp, Parch medians: [28.5  0.   0. ]
means after imputing:      [29.556  0.493  0.39 ]
standard deviations:       [13.016  1.06   0.838]
rows the scaler saw:       712.0
Age median, train only vs all 891 rows: 28.5 vs 28.0
Fare: median 14.45 · mean of log1p(Fare) 2.951
categories: [[1, 2, 3], ['female', 'male'], ['C', 'Q', 'S']]


**Resposta (depois de rodar):** O imputer aprendeu mediana de Age = **28.5**, não 28.0 —
confirmando a hipótese: ele só viu as 712 linhas de treino, diferente da mediana de todas as 891 linhas
completas (`X['Age'].median()` = 28.0). O scaler viu exatamente `n_samples_seen_ = 712.0` linhas, o mesmo
tamanho do treino. Médias pós-imputação: Age≈29.556, SibSp≈0.493, Parch≈0.39; desvios padrão: 13.016, 1.06,
0.838. A mediana de `Fare` é 14.45 e a média do `log1p(Fare)` é 2.951. Categorias aprendidas: Pclass
[1, 2, 3], Sex [female, male], Embarked [C, Q, S].

**Simulation: the preprocessing fit on all 891 passengers.** Reproduce the leak of section 4 on the Titanic: fit a copy of `preprocess` on all of `X` (use `clone` from `sklearn.base`), fit a `KNeighborsClassifier()` on the transformed training rows only, and score it on the transformed test rows. What median age does the copy learn? Does the test accuracy change? Then fit a copy of the whole `model` on all 891 rows and score it on the test rows.

In [7]:
# Simulation: reproduzir o vazamento ajustando o pre-processamento (e depois o pipeline todo) em todas as 891 linhas
from sklearn.base import clone

preprocess_leak = clone(preprocess)
preprocess_leak.fit(X)  # vazamento: viu as linhas de teste tambem
X_train_t = preprocess_leak.transform(X_train)
X_test_t = preprocess_leak.transform(X_test)
knn_leak = KNeighborsClassifier().fit(X_train_t, y_train)

age_median_leak = preprocess_leak.named_transformers_['num'].steps[0][1].statistics_[0]
age_median_honest = model.named_steps['prep'].named_transformers_['num'].steps[0][1].statistics_[0]
print(f"mediana de Age aprendida pela copia vazada (ajustada nas 891 linhas): {age_median_leak}")
print(f"mediana de Age aprendida dentro do pipeline honesto (so treino):     {age_median_honest}")
print(f"mediana de Age das 891 linhas completas (X['Age'].median()):         {X['Age'].median()}")
print()
print(f"acuracia no teste · pre-processamento vazado, KNN novo:  {knn_leak.score(X_test_t, y_test):.3f}")
print(f"acuracia no teste · pipeline honesto (referencia):       {model.score(X_test, y_test):.3f}")

model_leak = clone(model)
model_leak.fit(X, y)  # pipeline inteiro (pre-processamento + KNN) ajustado em todas as 891 linhas, y incluido
print(f"acuracia no teste · PIPELINE INTEIRO ajustado em todas as 891 linhas: {model_leak.score(X_test, y_test):.3f}")

mediana de Age aprendida pela copia vazada (ajustada nas 891 linhas): 28.0
mediana de Age aprendida dentro do pipeline honesto (so treino):     28.5
mediana de Age das 891 linhas completas (X['Age'].median()):         28.0

acuracia no teste · pre-processamento vazado, KNN novo:  0.782


acuracia no teste · pipeline honesto (referencia):       0.782


acuracia no teste · PIPELINE INTEIRO ajustado em todas as 891 linhas: 0.860


**Resposta:** A cópia do pré-processamento ajustada em todas as 891 linhas aprendeu mediana de
Age = **28.0** (igual a `X['Age'].median()`), contra **28.5** do pipeline honesto — o vazamento mudou o número
aprendido, mas a acurácia no teste ficou igual nos dois casos (**0.782**): o vazamento do imputer/scaler é
pequeno demais para mudar o resultado do k-NN, porque mediana/média/desvio são estatísticas agregadas que
quase não mudam ao trocar 179 linhas de um total de 891. Já ajustar o **pipeline inteiro** (inclusive o KNN)
em todas as 891 linhas, incluindo `y`, infla a acurácia para **0.860** — um salto de quase 8 pontos — porque
o próprio classificador k-NN passa a poder "ver" as linhas de teste como vizinhos de si mesmas (distância
zero) durante a predição. Os três números (0.782 pré-processamento vazado, 0.782 honesto, 0.860 pipeline
inteiro vazado) bateram exatamente com os publicados na página da aula.

**To close the fundamental part.** In one sentence each: (1) why is the bug that re-fits a scaler on the test set the most dangerous of the three? (2) What does `model.predict` do to each transformer, and why does that make the size of the batch irrelevant? (3) Why did fitting the scaler on all rows barely change the score, while the target encoder made it look about three times better?

**Resposta:**

1. **Por que o bug de reajustar o scaler no conjunto de teste é o mais perigoso dos três:** porque ele não
   quebra nada e quase não muda o número — a página relata MAE de \$42.814 contra \$43.023 do fluxo correto
   (praticamente indistinguível) quando aplicado ao conjunto de teste inteiro, mas piora bastante (\$65.754
   por lote regional, \$116.049 por distrito) quanto menor o lote reajustado. É o tipo de bug que passa
   despercebido em um teste malfeito e só explode em produção, quando os lotes de verdade são pequenos.
2. **O que `model.predict` faz com cada transformador, e por que o tamanho do lote é irrelevante:** cada
   transformador dentro do pipeline aplica só `transform` (nunca `fit`) sobre os parâmetros já aprendidos no
   treino — mediana, média, desvio, categorias — linha por linha, de forma independente. Confirmamos isso na
   prática abaixo: prever as 5 primeiras linhas de uma vez ou uma a uma dá exatamente a mesma resposta.
   Não existe nenhuma estatística recalculada a partir do lote de entrada, então 1 linha ou 1000 linhas dão o
   mesmo resultado por linha.
3. **Por que o scaler quase não mudou o resultado e o codificador de alvo "melhorou" o resultado em ~3×:** a
   página mostra que vazar o scaler muda o MAE em só uns \$4 em média, porque média e desvio padrão são
   estatísticas agregadas sobre milhares de linhas — trocar 20% delas (o conjunto de teste) quase não move a
   média. Já um codificador de alvo (*target encoding*) aprende, para cada categoria (zona), a média do
   próprio `y` daquele grupo; com 8.237 das 12.590 zonas tendo um único distrito, o "valor aprendido" da
   categoria é, na prática, a resposta certa já embutida — o MAE cai de \$42.552 (honesto) para \$14.567
   (vazado), quase 3× melhor, porque o modelo está respondendo de memória, não generalizando.

> Nota: os números desta resposta vêm do exemplo de regressão em habitação da própria página da aula (as 3
> "bugs" e o *target encoding* de zonas), que não faz parte deste notebook de Titanic — por isso não foram
> reexecutados aqui, só citados como referência para responder à pergunta conceitual.

**Checagem da pergunta 2 (o tamanho do lote é irrelevante):**

In [8]:
# Checagem para a pergunta 2 do "To close": o tamanho do lote importa?
pred_bulk = model.predict(X_test)
pred_one_by_one = np.array([model.predict(X_test.iloc[[i]])[0] for i in range(5)])
print("previsoes em lote (5 primeiras):   ", pred_bulk[:5])
print("previsoes uma a uma (5 primeiras): ", pred_one_by_one)
print("sao iguais?", np.array_equal(pred_bulk[:5], pred_one_by_one))

previsoes em lote (5 primeiras):   

 [0 0 0 0 1]
previsoes uma a uma (5 primeiras):  [0 0 0 0 1]
sao iguais? True


**Resposta:** Confirmado com o próprio pipeline do Titanic: as previsões das 5 primeiras
linhas de teste em lote (`model.predict(X_test)[:5]`) são idênticas, elemento a elemento, às previsões feitas
uma linha por vez (`model.predict(X_test.iloc[[i]])`) — `sao iguais? True`. Isso é a consequência direta de
cada transformador só aplicar `transform` com parâmetros fixos: nada no pipeline olha para o tamanho do lote
que está sendo transformado.

### Your turn (fundamental)


At home:

1. **By hand.** A column has training values 10, NaN, 30, 50 and new values NaN, 70. Compute the median, mean and standard deviation that `SimpleImputer(strategy='median')` followed by `StandardScaler()` learn, and the two transformed new values. Check with a pipeline. Then compute what `fit_transform` on the two new rows would give.
2. **Bug 1 with another scaler.** Repeat the regional-batch experiment of bug 1 with `MinMaxScaler` instead of `StandardScaler`, in both the model and the bug. Is the damage larger or smaller? Which statistic of a batch does min–max re-learn, and why is it more fragile than a mean?
3. **The size of a leak.** Repeat the target-encoding experiment of section 4 with zones rounded to one decimal (about 10 km) instead of two. How many zones have a single district now? Does the gap between the leaky and the honest MAE grow or shrink, and why?
4. **Break it on purpose.** Predict on `X_test` with its columns in reverse order; on `X_test.to_numpy()`; and on `X_test.drop(columns='Cabin')`. Which of the three work, which fail, and what in the pipeline explains each result?

**Item 1 — por conta própria, na mão:**

In [9]:
# Your turn (fundamental), item 1 - por conta propria: imputer + scaler, na mao
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

train_vals = np.array([[10.], [np.nan], [30.], [50.]])
new_vals = np.array([[np.nan], [70.]])

median_hand = np.nanmedian(train_vals)
imputed_train_hand = np.where(np.isnan(train_vals), median_hand, train_vals)
mean_hand = imputed_train_hand.mean()
std_hand = imputed_train_hand.std()  # populacional (ddof=0), igual ao StandardScaler

imputed_new_hand = np.where(np.isnan(new_vals), median_hand, new_vals)
scaled_new_hand = (imputed_new_hand - mean_hand) / std_hand

print("na mao -> mediana:", median_hand, "| media:", mean_hand, "| desvio padrao (populacional):", std_hand)
print("na mao -> novos valores imputados:", imputed_new_hand.ravel(), "| escalados:", scaled_new_hand.ravel())

pipe_hand = make_pipeline(SimpleImputer(strategy='median'), StandardScaler()).fit(train_vals)
print("conferindo com sklearn (fit no treino, transform nos novos):", pipe_hand.transform(new_vals).ravel())

trap_hand = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
print("a pegadinha -> fit_transform so nas duas linhas novas:", trap_hand.fit_transform(new_vals).ravel())

na mao -> mediana: 30.0 | media: 30.0 | desvio padrao (populacional): 14.142135623730951
na mao -> novos valores imputados: [30. 70.] | escalados: [0.         2.82842712]
conferindo com sklearn (fit no treino, transform nos novos): [0.         2.82842712]
a pegadinha -> fit_transform so nas duas linhas novas: [0. 0.]


**Resposta:** Com o treino 10, NaN, 30, 50 → mediana = **30.0** (ignorando o NaN); depois de
imputar, [10, 30, 30, 50] → média = **30.0**, desvio padrão populacional (o que `StandardScaler` usa,
`ddof=0`) = **14.14**. Com os novos valores NaN→30 (mediana do treino) e 70, os valores padronizados ficam
**[0.0, 2.83]** — exatamente o que o pipeline de verdade devolveu (`pipe_hand.transform`), confirmando a
conta na mão. A pegadinha de chamar `fit_transform` nas duas linhas novas sozinhas (em vez de `transform`) dá
**[0.0, 0.0]**: o `SimpleImputer`/`StandardScaler` reaprendem do zero só com [NaN, 70] — mediana 70, e depois
de imputar os dois valores ficam iguais (70, 70), então a variância é zero e o `StandardScaler` (que evita
dividir por zero) devolve 0 para os dois — o mesmo vazamento de "ajustar de novo nos dados novos" que a aula
inteira ensina a evitar.

**Resposta (Your turn fundamental, itens 2 e 3 — fora do escopo deste notebook de Titanic):**

- **Item 2 (bug 1 com `MinMaxScaler`):** não reproduzido aqui porque o dataset de habitação do "bug 1" (lotes
  regionais) não faz parte deste notebook de Titanic. Raciocinando sobre a mecânica: `MinMaxScaler` reaprende
  o mínimo e o máximo de cada lote — estatísticas de ordem (não agregadas como a média) — então um único
  valor extremo em um lote pequeno desloca a escala inteira. Eu esperaria um dano **maior** que com
  `StandardScaler` (cujo pior caso publicado foi \$116.049 por distrito), porque o mínimo/máximo de um lote de
  1 ou poucos distritos é muito mais instável — quase sempre "o próprio valor é o extremo" — do que uma
  média, que já é mais robusta mesmo em lotes pequenos.
- **Item 3 (zonas a 1 decimal, ~10 km, em vez de 2):** também não reproduzido aqui (mesmo dataset de
  habitação, fora do escopo deste lab). Células maiores agregam mais distritos por zona, então o número de
  zonas com um único distrito deve **cair** bastante em relação aos 8.237/12.590 publicados a 2 decimais — e,
  com menos zonas "memorizáveis", a diferença entre o MAE vazado (\$14.567) e o honesto (\$42.552) deve
  **encolher**, já que o codificador de alvo tem menos chance de devolver a resposta exata de um grupo de
  tamanho 1.

(Os itens 1 e 4 — a conta na mão e "quebrar de propósito" — foram implementados e respondidos em código
acima, dentro do escopo deste notebook.)

**Item 4 — quebrar de propósito:**

In [10]:
# Your turn (fundamental), item 4 - quebrar de proposito
X_test_reversed = X_test[X_test.columns[::-1]]
X_test_arr = X_test.to_numpy()
X_test_nocabin = X_test.drop(columns='Cabin')

ref_pred = model.predict(X_test)

try:
    p = model.predict(X_test_reversed)
    print("colunas em ordem invertida: OK -> mesmas previsoes do original?", np.array_equal(p, ref_pred))
except Exception as e:
    print("colunas em ordem invertida:", type(e).__name__, "->", str(e)[:200])

try:
    p = model.predict(X_test_arr)
    print("como array numpy: OK -> previu", len(p), "linhas")
except Exception as e:
    print("como array numpy:", type(e).__name__, "->", str(e)[:200])

try:
    p = model.predict(X_test_nocabin)
    print("sem a coluna Cabin: OK -> mesmas previsoes do original?", np.array_equal(p, ref_pred))
except Exception as e:
    print("sem a coluna Cabin:", type(e).__name__, "->", str(e)[:200])

colunas em ordem invertida: OK -> mesmas previsoes do original? True
como array numpy: ValueError -> Specifying the columns using strings is only supported for dataframes.
sem a coluna Cabin: OK -> mesmas previsoes do original? True


**Resposta:** Testado com o próprio pipeline do Titanic. Colunas em ordem invertida →
funciona e dá exatamente as mesmas previsões (`True`), porque o `ColumnTransformer` seleciona colunas por
**nome**, não por posição. Como array numpy (`X_test.to_numpy()`) → falha com
`ValueError: Specifying the columns using strings is only supported for dataframes.`, porque sem nomes de
coluna o `ColumnTransformer` não tem como localizar `'Age'`, `'Fare'` etc. Sem a coluna `Cabin` → funciona e
dá as mesmas previsões (`True`), porque nenhum branch usa `Cabin` (ela já era descartada pelo
`remainder='drop'`).

## Advanced part — steps 6 to 9

### Step 6 — what the model sees

**Predict.** How many columns reach the model: the 7 columns the branches use, or more? Will a third-class man have a 1 in one `Pclass` column or in several?

**Resposta (antes de rodar):** Mais que 7 colunas chegam ao modelo, porque o `OneHotEncoder` expande
cada categórica em uma coluna por categoria: `Pclass` (3 categorias) + `Sex` (2) + `Embarked` (3) = 8 colunas
binárias, mais as 3 numéricas (`Age`, `SibSp`, `Parch`) e 1 (`Fare`) = 12 colunas no total. Um homem de
3ª classe deve ter 1 em exatamente **uma** coluna de `Pclass` (a de valor "3") e 0 nas outras duas — one-hot é
mutuamente exclusivo por construção.

In [11]:
names = prep.get_feature_names_out()
print(len(names), "columns reach the model:", list(names))
prep.set_output(transform='pandas')
print(prep.transform(X_test.head(3)).round(2).T)

12 columns reach the model: ['num__Age', 'num__SibSp', 'num__Parch', 'fare__Fare', 'cat__Pclass_1', 'cat__Pclass_2', 'cat__Pclass_3', 'cat__Sex_female', 'cat__Sex_male', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S']
                  565   160   553
num__Age        -0.43  1.11 -0.58
num__SibSp       1.42 -0.47 -0.47
num__Parch      -0.47  0.73 -0.47
fare__Fare       0.28 -0.11 -0.86
cat__Pclass_1    0.00  0.00  0.00
cat__Pclass_2    0.00  0.00  0.00
cat__Pclass_3    1.00  1.00  1.00
cat__Sex_female  0.00  0.00  0.00
cat__Sex_male    1.00  1.00  1.00
cat__Embarked_C  0.00  0.00  1.00
cat__Embarked_Q  0.00  0.00  0.00
cat__Embarked_S  1.00  1.00  0.00


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


**Resposta (depois de rodar):** Chegam exatamente **12 colunas** ao modelo (as 4
numéricas/`Fare` + 8 do one-hot), confirmando a hipótese. E sim: nas três linhas de exemplo mostradas (todas
de 3ª classe), `cat__Pclass_3` vale 1.00 e `cat__Pclass_1`/`cat__Pclass_2` valem 0.00 — exatamente um 1 em
uma única coluna de `Pclass`, nunca em mais de uma.

### Step 7 — new passengers, raw

**Predict.** One new passenger boarded at a port the training data never saw, and another has no age. Will the pipeline crash, or answer? What number will replace the missing age?

**Resposta (antes de rodar):** Não deve travar. O `OneHotEncoder` do branch `cat` tem
`handle_unknown='ignore'`, então o porto desconhecido `'B'` deve virar uma linha de zeros nas três colunas de
`Embarked` (nenhuma delas liga). O `SimpleImputer` do branch `num` deve substituir a idade faltante pela
mediana aprendida no treino — a mesma mediana do Predict do Step 5, não necessariamente 28.0.

In [12]:
new = pd.DataFrame({
    'PassengerId': [901, 902, 903], 'Pclass': [3, 1, 2],
    'Name': ['Doe, Mr. John', 'Roe, Mrs. Jane', 'Poe, Miss. Ann'],
    'Sex': ['male', 'female', 'female'], 'Age': [25, np.nan, 30],
    'SibSp': [0, 1, 0], 'Parch': [0, 0, 0], 'Ticket': ['A1', 'B2', 'C3'],
    'Fare': [8.05, 80.0, 15.0], 'Cabin': [np.nan, 'C85', np.nan],
    'Embarked': ['S', 'C', 'B'],             # 'B' (Belfast) never appears in the training data
})
print(prep.transform(new)[['num__Age', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S']].round(2))
print("P(survived):", model.predict_proba(new)[:, 1].round(2), "· prediction:", model.predict(new))

   num__Age  cat__Embarked_C  cat__Embarked_Q  cat__Embarked_S
0     -0.35              0.0              0.0              1.0
1     -0.08              1.0              0.0              0.0
2      0.03              0.0              0.0              0.0


P(survived): [0. 1. 1.] · prediction: [0 1 1]


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebo

**Resposta (depois de rodar):** Não travou. A passageira com `Embarked='B'` (porto nunca visto)
ficou com **0.00 nas três colunas** `cat__Embarked_{C,Q,S}` — a linha de zeros prevista pelo
`handle_unknown='ignore'`. O passageiro sem idade (`Age=NaN`) recebeu a idade imputada equivalente à mediana
do treino (refletida no valor padronizado `num__Age = -0.08`, não um valor arbitrário). As previsões finais
foram `P(sobreviveu) = [0.0, 1.0, 1.0]` e classes `[0, 1, 1]`.

**Simulation: an encoder that refuses new categories.** Make a copy of `model` (with `clone`) whose one-hot encoder of the `'cat'` branch has `handle_unknown='error'`; the path is `prep__cat__onehotencoder__handle_unknown`, set with `set_params`. Fit it on the training rows and predict the first two new passengers, then all three. What happens, and what would it mean for a service that answers one request at a time?

In [13]:
# Simulation: um encoder que recusa categorias novas
from sklearn.base import clone

model_strict = clone(model)
model_strict.set_params(prep__cat__onehotencoder__handle_unknown='error')
model_strict.fit(X_train, y_train)

try:
    pred2 = model_strict.predict(new.iloc[:2])
    print("primeiros dois novos passageiros: OK ->", pred2)
except ValueError as e:
    print("primeiros dois novos passageiros: ValueError ->", str(e)[:200])

try:
    pred3 = model_strict.predict(new)
    print("os tres novos passageiros: OK ->", pred3)
except ValueError as e:
    print("os tres novos passageiros: ValueError ->", str(e)[:200])

primeiros dois novos passageiros: OK -> [0 1]
os tres novos passageiros: ValueError -> Found unknown categories ['B'] in column 2 during transform


**Resposta:** Com `handle_unknown='error'`, prever os dois primeiros novos passageiros funciona
normalmente (`[0, 1]`), porque nenhum deles tem uma categoria desconhecida. Mas prever os três ao mesmo tempo
falha com `ValueError: Found unknown categories ['B'] in column 2 during transform` — a 3ª passageira
embarcou em 'B', categoria nunca vista no treino. Para um serviço que responde um pedido por vez, isso é
perigoso: um único cliente com um valor "estranho" (um porto novo, uma categoria rara) derruba a requisição
inteira com uma exceção não tratada, em vez de simplesmente receber uma previsão (ainda que menos informada)
como `handle_unknown='ignore'` permitiria.

### Step 8 — save, load, compare

**Predict.** Will the saved file be a few kilobytes or hundreds? Will the loaded model give exactly the same probabilities, or only approximately?

**Resposta (antes de rodar):** Espero um arquivo de **centenas de KB**, não poucos KB: um k-NN
guarda dentro de si os dados de treino inteiros já transformados (712 linhas), então o pipeline salvo carrega
esses dados, não só alguns parâmetros. As probabilidades do modelo carregado devem ser **exatamente** iguais
às do original (não apenas aproximadas) — `joblib` serializa o estado interno completo (arrays, estatísticas
aprendidas), sem nenhum refit nem fonte de aleatoriedade ao recarregar.

In [14]:
import os
import joblib

joblib.dump(model, 'titanic-knn.joblib')
loaded = joblib.load('titanic-knn.joblib')
print(f"{os.path.getsize('titanic-knn.joblib') / 1e3:.0f} kB")
print("same test predictions:", np.array_equal(loaded.predict(X_test), model.predict(X_test)))
print("same probabilities for the new rows:",
      np.array_equal(loaded.predict_proba(new), model.predict_proba(new)))

162 kB


same test predictions: True
same probabilities for the new rows: True


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/

**Resposta (depois de rodar):** O arquivo pesa **162 kB** — na faixa de "centenas", como eu
esperava (não poucos KB), porque o k-NN carrega as 712 linhas de treino transformadas dentro de si. As
previsões do modelo recarregado são **exatamente** iguais às do original: `same test predictions: True` e
`same probabilities for the new rows: True` — sem nenhuma diferença, nem mesmo de arredondamento.

### Step 9 — a transformer of your own: the title

**Predict.** Should the list of "frequent" titles be learned on the training rows, or can it be written by hand from all 891 passengers? Will the title improve the accuracy, and are 179 test passengers enough to be sure?

**Resposta (antes de rodar):** A lista de títulos frequentes deve ser aprendida **só no treino**
(712 linhas) — escrevê-la à mão a partir das 891 linhas completas vazaria informação do teste para dentro do
"vocabulário" que o pipeline usa, exatamente o tipo de vazamento que esta aula inteira existe para evitar.
Espero que o título melhore um pouco a acurácia, porque carrega informação de sexo/idade/status social que não
está 100% capturada pelas outras colunas (ex.: "Master" sinaliza menino). Mas com apenas 179 passageiros de
teste, uma melhora de poucos pontos percentuais corresponde a poucos passageiros a mais acertados — não é o
suficiente para ter certeza estatística sem uma validação cruzada.

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin

class TitleExtractor(TransformerMixin, BaseEstimator):
    """Name -> title ('Mr', 'Mrs', ...); titles rarer than min_count in TRAIN become 'Rare'."""
    def __init__(self, min_count=10):
        self.min_count = min_count

    def _titles(self, X):
        names = pd.Series(np.asarray(X).ravel())
        return names.str.extract(r',\s*([^.]+)\.')[0].str.strip()

    def fit(self, X, y=None):
        counts = self._titles(X).value_counts()
        self.titles_ = sorted(counts[counts >= self.min_count].index)   # learned on train
        return self

    def transform(self, X):
        t = self._titles(X)
        return t.where(t.isin(self.titles_), 'Rare').to_frame('title')

    def get_feature_names_out(self, input_features=None):
        return np.array(['title'])

prep2 = ColumnTransformer(preprocess.transformers + [
    ('title', make_pipeline(TitleExtractor(), OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
     ['Name'])])
model2 = Pipeline([('prep', prep2), ('knn', KNeighborsClassifier())]).fit(X_train, y_train)
print("titles learned on train:", model2['prep'].named_transformers_['title'][0].titles_)
print("titles of the new rows: ", TitleExtractor().fit(X_train[['Name']]).transform(new[['Name']])['title'].to_list())
print(f"test accuracy · without title {model.score(X_test, y_test):.3f} · with title {model2.score(X_test, y_test):.3f}")

titles learned on train:

 ['Master', 'Miss', 'Mr', 'Mrs']
titles of the new rows:  ['Mr', 'Mrs', 'Miss']


test accuracy · without title 0.782 · with title 0.816


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


**Resposta (depois de rodar):** A lista de títulos foi aprendida com
`TitleExtractor().fit(X_train[['Name']])`, só no treino — confirmando a hipótese de que vazar as 891 linhas
completas seria o erro que a aula toda evita. Os títulos aprendidos no treino são
`['Master', 'Miss', 'Mr', 'Mrs']`, e a acurácia subiu de 0.782 (sem título) para **0.816** (com título) — uma
melhora de 3.4 pontos percentuais. Com 179 passageiros de teste, 3.4 pontos correspondem a só ~6 passageiros
classificados diferentemente — poucos demais para confiar sem uma validação cruzada, exatamente como eu
esperava.

**Simulation: how rare is "Rare"?** `min_count` is a hyperparameter of `TitleExtractor`, so the pipeline reaches it as `prep__title__titleextractor__min_count`. For `min_count` in 1, 5, 10 and 40, fit a copy of `model2` (with `clone` and `set_params`) and print the titles it learned and its test accuracy. Which titles survive each threshold, and does the accuracy say which threshold is best?

In [16]:
# Simulation: quao raro e "Rare"?
from sklearn.base import clone

for mc in [1, 5, 10, 40]:
    m = clone(model2)
    m.set_params(prep__title__titleextractor__min_count=mc)
    m.fit(X_train, y_train)
    titles = m.named_steps['prep'].named_transformers_['title'][0].titles_
    acc = m.score(X_test, y_test)
    print(f"min_count={mc:2d} -> {len(titles):2d} titulos {titles} | acuracia no teste {acc:.3f}")

min_count= 1 -> 14 titulos ['Col', 'Don', 'Dr', 'Jonkheer', 'Lady', 'Major', 'Master', 'Miss', 'Mlle', 'Mr', 'Mrs', 'Ms', 'Rev', 'Sir'] | acuracia no teste 0.810
min_count= 5 ->  6 titulos ['Dr', 'Master', 'Miss', 'Mr', 'Mrs', 'Rev'] | acuracia no teste 0.816


min_count=10 ->  4 titulos ['Master', 'Miss', 'Mr', 'Mrs'] | acuracia no teste 0.816


min_count=40 ->  3 titulos ['Miss', 'Mr', 'Mrs'] | acuracia no teste 0.816


**Resposta:** `min_count=1` aprende 14 títulos distintos (acurácia 0.810); a partir de
`min_count=5` o conjunto já se estabiliza em poucos títulos (6, depois 4 em `min_count=10`, depois 3 em
`min_count=40`), e a acurácia fica em **0.816 para 5, 10 e 40** — exatamente igual nos três. Isso quer dizer
que, com apenas 179 passageiros de teste, a acurácia não consegue distinguir qual desses três limiares é
"melhor": a diferença está dentro do ruído estatístico do conjunto de teste, e escolher entre eles exigiria
validação cruzada (como o "Your turn" avançado faz com `k` mais abaixo).

**To close the lesson.** In one sentence each: (1) why save the whole pipeline and not only the trained classifier? (2) When must a step of your own be a class rather than a `FunctionTransformer`? (3) Why does a search that tries settings through `set_params` never let the scoring rows into the preprocessing?

**Resposta:**

1. **Por que salvar o pipeline inteiro, e não só o classificador treinado:** porque o classificador por si só
   não sabe o que fazer com uma linha crua (`Age=NaN`, `Sex='male'`, `Embarked='B'`) — ele só entende números
   já imputados, escalados e codificados. Salvar o `Pipeline` guarda também as medianas, médias, desvios e
   categorias aprendidas no treino, então carregar o arquivo e chamar `.predict` numa linha nova funciona sem
   precisar reescrever (e arriscar errar) o pré-processamento à mão.
2. **Quando um passo precisa ser uma classe, e não um `FunctionTransformer`:** quando ele precisa **aprender**
   algo no treino para repetir depois no teste — como a lista de títulos frequentes do `TitleExtractor`, ou os
   percentis do `Clipper` usado acima. Um `FunctionTransformer` não guarda estado aprendido entre `fit` e
   `transform`; sem uma classe com os dois métodos separados, não haveria onde armazenar "o que foi aprendido
   no treino" para reaplicar no teste.
3. **Por que uma busca via `set_params` nunca deixa as linhas de avaliação entrarem no pré-processamento:**
   porque `set_params` só troca hiperparâmetros (como `knn__n_neighbors` ou
   `prep__title__titleextractor__min_count`) — ele não decide quais linhas são usadas para `fit`. Quem chama
   `.fit(X_train, ...)` depois do `set_params` continua escolhendo explicitamente só as linhas de treino; a
   arquitetura do pipeline nunca dá ao buscador acesso direto às linhas de validação/teste para ajustar
   estatística alguma — o mesmo motivo pelo qual a Simulação do Step 5 precisou *forçar* o vazamento
   manualmente (`clone(preprocess).fit(X)`) para conseguir reproduzi-lo.

### Your turn (advanced)


At home, with more time:

1. **Ratios as a step.** Rooms per household, bedrooms per room and people per household are more meaningful than raw counts. Write them as a `FunctionTransformer` (use a function that receives a DataFrame and returns one with the three new columns, and `feature_names_out` as a function that returns their names), put it in the numeric branch of section 3, and compare the test MAE with the lesson's \$41,552.
2. **The deck.** The first letter of `Cabin` is the deck (A to G, T), and a missing cabin is information too (mostly third class). Add a branch that turns `Cabin` into the deck, with `'U'` for missing, followed by one-hot encoding. Write the conversion first as a `lambda` inside a `FunctionTransformer`, fit, and try `joblib.dump` on the model. What happens, and why does a function defined with `def` fix it?
3. **Choose `k` honestly.** Carve a validation split out of `X_train` (25% of it, stratified). For `k` in 1, 5, 15, 45, use `model.set_params(knn__n_neighbors=k)`, fit on the rest of the training rows and score on the validation rows. Pick the best `k`, refit on all of `X_train`, and only then look at the test accuracy, once. (The [cross-validation](https://insper.github.io/ML/2026.2/classes/validation/) lesson will make this choice more robust.)
4. **Clip the fare.** Replace `log1p` in the `'fare'` branch with the lesson's `Clipper` (percentiles 1 and 99). Print its `upper_` and compare it with the largest fare in the training rows. Does the test accuracy change? Would you expect k-NN to be more sensitive to the fare's outliers than a model that only compares a value to a threshold?

**Resposta (Your turn avançado, item 1 — fora do escopo deste notebook de Titanic):**

**Item 1 (razões como feature, dataset de habitação):** não reproduzido aqui — esse exercício usa o dataset
de preços de imóveis da Califórnia (`total_rooms`, `households` etc.), que não faz parte deste lab de
Titanic. Conceitualmente, `rooms_per_household`, `bedrooms_per_room` e `population_per_household` tendem a
ajudar porque "cômodos totais" por si só mistura o tamanho do quarteirão censitário com o tamanho típico de
cada casa; a razão isola o padrão por domicílio, mais comparável de bairro para bairro. A referência da aula
é o MAE de **\$41.552** do `ColumnTransformer` com one-hot — eu esperaria uma mudança pequena (para melhor),
não uma queda drástica no MAE, já que k-NN/regressores de vizinhos próximos ganham mais com escalas
consistentes do que com a engenharia exata de cada razão.

(Os itens 2, 3 e 4 — o deck, escolher `k` honestamente e recortar a tarifa — foram implementados e
respondidos em código abaixo, dentro do escopo deste notebook.)

**Item 2 — o deck (`Cabin` -> letra do deck), `lambda` vs `def`:**

In [17]:
# Your turn (advanced), item 2 - o deck, lambda vs def
import joblib

def cabin_to_deck(X):
    s = pd.DataFrame(X).iloc[:, 0]
    return s.str[0].fillna('U').to_frame('Cabin')  # 'one-to-one' exige o mesmo nome da coluna de entrada

deck_lambda = FunctionTransformer(lambda X: pd.DataFrame(X).iloc[:, 0].str[0].fillna('U').to_frame('Cabin'),
                                   feature_names_out='one-to-one')
deck_def = FunctionTransformer(cabin_to_deck, feature_names_out='one-to-one')

prep3_lambda = ColumnTransformer(preprocess.transformers + [
    ('deck', make_pipeline(deck_lambda, OneHotEncoder(handle_unknown='ignore', sparse_output=False)), ['Cabin']),
])
model3_lambda = Pipeline([('prep', prep3_lambda), ('knn', KNeighborsClassifier())]).fit(X_train, y_train)
print("acuracia no teste, deck via lambda:", f"{model3_lambda.score(X_test, y_test):.3f}")
try:
    joblib.dump(model3_lambda, 'titanic-knn-deck-lambda.joblib')
    print("joblib.dump com lambda: OK")
except Exception as e:
    print("joblib.dump com lambda:", type(e).__name__, "->", str(e)[:200])

prep3_def = ColumnTransformer(preprocess.transformers + [
    ('deck', make_pipeline(deck_def, OneHotEncoder(handle_unknown='ignore', sparse_output=False)), ['Cabin']),
])
model3_def = Pipeline([('prep', prep3_def), ('knn', KNeighborsClassifier())]).fit(X_train, y_train)
print("acuracia no teste, deck via def:   ", f"{model3_def.score(X_test, y_test):.3f}")
try:
    joblib.dump(model3_def, 'titanic-knn-deck-def.joblib')
    print("joblib.dump com def: OK, tamanho:", f"{os.path.getsize('titanic-knn-deck-def.joblib')/1e3:.0f} kB")
except Exception as e:
    print("joblib.dump com def:", type(e).__name__, "->", str(e)[:200])

acuracia no teste, deck via lambda: 0.793
joblib.dump com lambda: PicklingError -> Can't pickle <function <lambda> at 0x7fdf29627060>: it's not found as __main__.<lambda>
acuracia no teste, deck via def:    0.793
joblib.dump com def: OK, tamanho: 133 kB


**Resposta:** As duas versões (lambda e `def`) chegam à mesma acurácia de teste, **0.793**
— a função em si não muda o que é aprendido, só como ela é armazenada. Mas `joblib.dump` da versão com
`lambda` falha com `PicklingError: Can't pickle <function <lambda> ...>: it's not found as __main__.<lambda>`,
porque o módulo `pickle` (usado por baixo dos panos pelo `joblib`) só sabe serializar uma função pelo caminho
`módulo.nome` — e uma `lambda` não tem um nome importável. A versão com `def cabin_to_deck(X): ...` salva sem
problema (**133 kB**), porque uma função `def` definida no nível do módulo tem um nome que o `pickle`
consegue referenciar e reimportar depois.

**Item 3 — escolher `k` honestamente (split de validação):**

In [18]:
# Your turn (advanced), item 3 - escolher k honestamente
from sklearn.base import clone

X_tr2, X_val, y_tr2, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=42)
print(f"treino reduzido: {X_tr2.shape[0]} linhas | validacao: {X_val.shape[0]} linhas")

val_scores = {}
for k in [1, 5, 15, 45]:
    m = clone(model)
    m.set_params(knn__n_neighbors=k)
    m.fit(X_tr2, y_tr2)
    val_scores[k] = m.score(X_val, y_val)
    print(f"k={k:2d} -> acuracia de validacao {val_scores[k]:.3f}")

best_k = max(val_scores, key=val_scores.get)
print(f"melhor k na validacao: {best_k} (acuracia {val_scores[best_k]:.3f})")

model_bestk = clone(model)
model_bestk.set_params(knn__n_neighbors=best_k)
model_bestk.fit(X_train, y_train)  # agora sim, re-treina em TODO o X_train
print(f"acuracia no teste, k={best_k}, treinado em todo X_train (olhado so uma vez): {model_bestk.score(X_test, y_test):.3f}")
print(f"acuracia no teste, k=5 (default, para comparacao):                          {model.score(X_test, y_test):.3f}")

treino reduzido: 534 linhas | validacao: 178 linhas


k= 1 -> acuracia de validacao 0.747


k= 5 -> acuracia de validacao 0.843


k=15 -> acuracia de validacao 0.831


k=45 -> acuracia de validacao 0.758
melhor k na validacao: 5 (acuracia 0.843)


acuracia no teste, k=5, treinado em todo X_train (olhado so uma vez): 0.782


acuracia no teste, k=5 (default, para comparacao):                          0.782


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


**Resposta:** Separei 25% de `X_train` como validação (534 linhas de treino / 178 de
validação, estratificado). Resultados de validação: k=1 → 0.747, k=5 → **0.843** (melhor), k=15 → 0.831,
k=45 → 0.758 — um padrão clássico de sub/sobreajuste (k muito baixo é ruidoso demais, k muito alto borra as
classes). Depois de escolher k=5 e re-treinar em todo o `X_train`, a acurácia de teste (olhada **uma única
vez**) foi 0.782 — igual à do k=5 padrão usado durante toda a aula, o que sugere que o default já era, por
coincidência, a melhor escolha neste conjunto.

**Item 4 — recortar a tarifa com `Clipper`:**

In [19]:
# Your turn (advanced), item 4 - recortar a tarifa (Clipper)
from sklearn.base import BaseEstimator, TransformerMixin

class Clipper(TransformerMixin, BaseEstimator):
    """Recorta cada coluna nos percentis aprendidos no treino (lower_q, upper_q)."""
    def __init__(self, lower_q=1, upper_q=99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=float)
        self.lower_ = np.percentile(arr, self.lower_q, axis=0)
        self.upper_ = np.percentile(arr, self.upper_q, axis=0)
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=float)
        return np.clip(arr, self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features)

preprocess_clip = ColumnTransformer([
    ('num',  make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
             ['Age', 'SibSp', 'Parch']),
    ('fare', make_pipeline(SimpleImputer(strategy='median'), Clipper(1, 99), StandardScaler()),
             ['Fare']),
    ('cat',  make_pipeline(SimpleImputer(strategy='most_frequent'),
                           OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
             ['Pclass', 'Sex', 'Embarked']),
])
model_clip = Pipeline([('prep', preprocess_clip), ('knn', KNeighborsClassifier())]).fit(X_train, y_train)

clipper_fit = model_clip.named_steps['prep'].named_transformers_['fare'].steps[1][1]
print("upper_ aprendido pelo Clipper (percentil 99 do Fare de treino):", clipper_fit.upper_)
print("maior Fare no treino (bruto):                                 ", X_train['Fare'].max())
print(f"acuracia no teste · com log1p (original) {model.score(X_test, y_test):.3f} · com Clipper {model_clip.score(X_test, y_test):.3f}")

upper_ aprendido pelo Clipper (percentil 99 do Fare de treino): [227.525]
maior Fare no treino (bruto):                                  512.3292
acuracia no teste · com log1p (original) 0.782 · com Clipper 0.799


/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/emilybrtt/Insper/4-semestre/machine-learning/notebooks-ml/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


**Resposta:** O `Clipper(1, 99)` aprendeu `upper_ = 227.53` no treino, bem abaixo da maior
tarifa bruta do treino (**512.33**) — ou seja, ele recorta um punhado de tarifas extremas para esse teto. A
acurácia de teste subiu de 0.782 (com `log1p`) para **0.799** (com `Clipper`), uma pequena melhora. Isso é
coerente com a expectativa: k-NN mede distância euclidiana diretamente, então um valor extremo de `Fare`
(mesmo depois de escalado) pode dominar a distância a um vizinho; um classificador baseado em limiares (como
uma árvore) compara o valor a um corte e seria bem menos sensível a esse mesmo outlier.

## Conclusões

- O pipeline do Titanic (imputação + escala/log + one-hot + k-NN) atingiu **0.782** de acurácia no teste,
  contra **0.615** da baseline "sempre não sobreviveu" — e sem os `StandardScaler`s essa acurácia cai para
  **0.676**, porque a `Fare` bruta (até 512.33) domina a distância do k-NN sobre `Age`/`SibSp`/`Parch`.
- A mediana de `Age` aprendida DENTRO do pipeline foi **28.5** (só as 712 linhas de treino), diferente da
  mediana das 891 linhas completas (**28.0**) — prova direta de que nenhuma estatística do pipeline "espiou"
  o conjunto de teste.
- Vazar o pré-processamento (ajustá-lo em todas as 891 linhas) quase não mudou a acurácia (0.782 → 0.782),
  mas vazar o pipeline INTEIRO (incluindo o `y` de teste no treino do k-NN) inflou a acurácia para **0.860** —
  mostrando que o tamanho de um vazamento depende do que exatamente é aprendido, não apenas de "ter visto"
  dados demais.
- Extrair o título do nome (`Mr`/`Mrs`/`Miss`/`Master`, aprendido só no treino) subiu a acurácia de 0.782 para
  **0.816**, mas essa melhora de 3.4 pontos corresponde a só ~6 dos 179 passageiros de teste — pouco para
  confiar sem validação cruzada, confirmado pela simulação de `min_count` (0.810 / 0.816 / 0.816 / 0.816 para
  1 / 5 / 10 / 40).
- O pipeline salvo com `joblib` pesa **162 kB** e recarrega com previsões **exatamente** idênticas; mas uma
  função `lambda` dentro de um `FunctionTransformer` quebra esse salvamento (`PicklingError`), enquanto uma
  função `def` no nível do módulo funciona (133 kB) — uma armadilha comum ao escrever transformadores
  customizados.
- Escolher `k` com uma validação de 178 linhas (25% do treino) apontou k=5 como melhor (0.843 de validação)
  dentre {1, 5, 15, 45}; usado para refit em todo o treino e avaliado **uma única vez** no teste, deu 0.782 —
  igual ao k=5 default, reforçando que a escolha de hiperparâmetro deve vir de dados de validação, nunca do
  próprio conjunto de teste.